# Recursive tree map

Developing a treemap optimization template. 

In [ ]:
import pathlib

import networkx as nx
from matplotlib import pyplot as plt

from vizopt import introspection
from vizopt.templates.trees.recursive_raster_treemap import RasterTreemapOptimizer

### Simplest tree example

In [ ]:
simple_tree = nx.DiGraph()
simple_tree.add_edges_from([(0, 1), (0, 2), (1, 3), (3, 4), (3, 5)])
simple_tree.add_node(4, size=1)
simple_tree.add_node(5, size=2)
simple_tree.add_node(2, size=3)
_, ax = plt.subplots(figsize=(4, 3))
nx.draw(simple_tree, with_labels=True, ax=ax)

In [ ]:
sizes = introspection.compute_subtree_sizes(simple_tree, root=0)

treemap_optimizer = RasterTreemapOptimizer(
    simple_tree,
    sizes,
    grid_resolution=96,
    n_iters=2500,
    learning_rate=0.01,
)
treemap_optimizer.optimize()
len(treemap_optimizer.node_shapes_)

In [ ]:
ax = treemap_optimizer.plot()
ax.set_title("Recursive raster treemap of src/vizopt (RasterTreemapOptimizer)")
plt.gcf().set_size_inches(6, 6)
plt.show()

### Custom boundary representation

`RasterTreemapOptimizer` defaults to `Discrete` (one radius per angle). Passing a `representation` swaps in a different boundary parametrization — shared across every level of the recursion — e.g. `Fourier` for smooth, C∞ boundaries from a compact per-node coefficient vector.

In [ ]:
from vizopt.components.stars import BSpline

treemap_optimizer_fourier = RasterTreemapOptimizer(
    simple_tree,
    sizes,
    representation=BSpline(n_ctrl_pts=16, k_angles=128), # representation=Fourier(k_angles=64, n_harmonics=6),
    grid_resolution=24,
    n_iters=2500,
    learning_rate=0.01,
)
treemap_optimizer_fourier.optimize()

ax = treemap_optimizer_fourier.plot()
ax.set_title("Recursive raster treemap with Fourier boundary representation")
plt.gcf().set_size_inches(6, 6)
plt.show()

### Testing tree: the file/directory tree of `src/vizopt` itself.

In [ ]:
src_dir = pathlib.Path().resolve().parents[1] / "src" / "vizopt"
src_dir

In [ ]:
file_tree = introspection.build_file_tree(src_dir)
nx.is_arborescence(file_tree), file_tree.number_of_nodes(), file_tree.number_of_edges()

## Baseline: squarified treemap heuristic

`introspection.plot_treemap` already gives a deterministic (non-optimized) layout via `squarify_layout`, using each file's byte size as its weight. Useful as a reference/initializer for the new template.

In [ ]:
introspection.plot_treemap(file_tree, padding=0.025)

## `RasterTreemapOptimizer`

Recursive raster-based star-domain treemap, promoted to `vizopt.templates.trees.recursive_raster_treemap.RasterTreemapOptimizer` (with unit tests in `tests/test_recursive_raster_treemap.py`) after developing it interactively against this tree: each node's children are jointly fit with `RasterStarOptimizer` (raster collision for mutual exclusion, an analytic containment term against the node's own already-fitted boundary, and a compactness term to close the whitespace exclusion+area+perimeter alone would leave behind), then recursed into depth-first using each child's *achieved* area as the next level's container budget.

Two generalizations made while extracting it:

- `star_polygon_area` and `radius_at_angle` moved to `vizopt.components.stars` as public numpy utilities — the same area formula was duplicated inline in this exploration, in `raster_based.ipynb`'s British Isles cell, and in the JAX `_multi_term_area` term.
- Child-rectangle seeding now calls `vizopt.treemap.squarify_layout` directly instead of `introspection.treemap_layout`, since the latter is file-tree-specific (it branches on a `graph.nodes[child]["is_dir"]` attribute that only `build_file_tree` graphs carry). The module works on any tree given a `sizes` dict, not just file trees.

In [ ]:
sizes = introspection.compute_subtree_sizes(file_tree)

treemap_optimizer = RasterTreemapOptimizer(
    file_tree,
    sizes,
    grid_resolution=96,
    n_iters=2500,
    learning_rate=0.01,
)
treemap_optimizer.optimize()
len(treemap_optimizer.node_shapes_)

In [ ]:
ax = treemap_optimizer.plot()
ax.set_title("Recursive raster treemap of src/vizopt (RasterTreemapOptimizer)")
plt.gcf().set_size_inches(10, 10)
plt.show()

## Second example: glottolog language-family forest

`notebooks/data/glotto_subgraph.json` (174 nodes, 161 edges) is a genuine *forest*, not a single tree: 13 language families, each its own root, no attribute coupling to file trees (leaf nodes carry `size`, not `is_dir`/byte counts). It's the case we explicitly decided to allow but hadn't exercised yet.

`RasterTreemapOptimizer` still expects one root, so the forest is reduced to a tree the standard way: add a virtual super-root with an edge to each of the 13 family roots. No changes needed to the module — `root` was already an explicit constructor override.

In [ ]:
import json

from networkx.readwrite import json_graph

data_dir = pathlib.Path().resolve().parents[1] / "notebooks" / "data"
with open(data_dir / "glotto_subgraph.json", encoding="utf-8") as f:
    glotto_tree = json_graph.node_link_graph(json.load(f), directed=True)

nx.is_forest(glotto_tree), glotto_tree.number_of_nodes(), glotto_tree.number_of_edges()

In [ ]:
glotto_roots = [n for n in glotto_tree.nodes if glotto_tree.in_degree(n) == 0]
glotto_root = "__root__"
glotto_tree.add_node(glotto_root)
glotto_tree.add_edges_from((glotto_root, r) for r in glotto_roots)

nx.is_arborescence(glotto_tree), len(glotto_roots)

In [ ]:
from vizopt.treemap import subtree_sizes

glotto_sizes = subtree_sizes(glotto_tree, glotto_root)
glotto_sizes[glotto_root]

In [ ]:
glotto_optimizer = RasterTreemapOptimizer(
    glotto_tree,
    glotto_sizes,
    root=glotto_root,
    grid_resolution=48,
    n_iters=2000,
    learning_rate=0.01,
)
glotto_optimizer.optimize()
len(glotto_optimizer.node_shapes_)

In [ ]:
ax = glotto_optimizer.plot(label_fn=lambda n: glotto_tree.nodes[n].get("name", str(n)))
ax.set_title("Recursive raster treemap of the glottolog language-family forest")
plt.gcf().set_size_inches(14, 14)
plt.show()